# Phase 3: Baseline — Raw Qwen3-4B with Tools

Run the **un-fine-tuned** Qwen3-4B with tool calling enabled to establish "before" metrics.
This gives us failure examples for the talk and a baseline reward score to compare against SFT and RL.

**Expected failures:** wrong tool selection, malformed JSON, infinite looping, no final output, no reasoning.

## Local MLX Testing (Mac)

Before running the full baseline on Databricks with Unsloth, we can test inference locally using
**mlx-lm** on Apple Silicon. This validates our agent loop and tool calling against a local model.

### Prerequisites (run in terminal)

**1. Install mlx-lm and login to HuggingFace:**
```bash
uv add "mlx-lm[train]"
uv run huggingface-cli login
```

**2. Quick CLI test (no server needed):**
```bash
uv run mlx_lm.generate \
  --model mlx-community/Qwen3.5-4B-MLX-4bit \
  --prompt "What are Apple's main revenue segments?" \
  --max-tokens 500
```

**3. Start the local server:**
```bash
# Single-request testing
uv run mlx_lm.server --model mlx-community/Qwen3.5-4B-MLX-4bit --port 8080

# Batch baseline (concurrent agent runs)
uv run mlx_lm.server \
  --model mlx-community/Qwen3.5-4B-MLX-4bit \
  --port 8080 \
  --prompt-concurrency 8 \
  --decode-concurrency 32 \
  --prompt-cache-size 4
```

> `--prompt-concurrency` controls parallel prefill (default 8), `--decode-concurrency` controls
> parallel token generation (default 32). `--prompt-cache-size` caps KV caches in memory
> (default 10 — too high for 16GB Mac, causes OOM). Set to 4 for Apple Silicon.

**4. Verify with curl:**
```bash
curl http://localhost:8080/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"messages": [{"role": "user", "content": "Say hello in 5 words"}], "max_tokens": 200}'
```

Once the server is running on port 8080, run the cells below.

In [1]:
# MLX Local Test — requires mlx_lm.server running on port 8080
# Start it first: uv run mlx_lm.server --model mlx-community/Qwen3.5-4B-MLX-4bit --port 8080

import json
import sys
from pathlib import Path

from openai import AsyncOpenAI

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "nb"))

from bbb.agent import run_tool_calling_agent_chat, TOOL_SCHEMAS_CHAT
from bbb.helpers__data_gen import SYSTEM_PROMPT
from bbb.tools import TOOL_FUNCTIONS
from bbb.helpers__inference import compute_reward

client = AsyncOpenAI(base_url="http://localhost:8080/v1", api_key="none")
VALID_TOOL_NAMES = set(TOOL_FUNCTIONS.keys())

Simple test

In [ ]:
result = await run_tool_calling_agent_chat(
    client=client,
    model="default_model",
    user_prompt="Briefly, what is AAPL's main revenue segment?",
    system_prompt="You are a helpful assistant. Be concise.",
    max_iterations=5,
    max_tokens=4096,
)

In [3]:
print(result['output_text'])

<think>
The user is asking about Apple Inc. (AAPL)'s main revenue segment.

1.  **Identify the company:** AAPL is Apple Inc.
2.  **Recall revenue categories:** Apple's revenue is generally categorized into four segments: Products, Services, and sometimes combined into broader groupings in earnings reports (e.g., iPhone, Mac, iPad, Wearables, Home & Accessories, Services).
3.  **Determine the largest segment:** Historically and consistently, the "iPhone" category is the largest contributor to Apple's total revenue. Even when grouped under "Products," the "iPhone" sub-category is the dominant driver.
4.  **Formulate the answer:** State clearly that "Products" (specifically iPhones) is the main segment.
5.  **Refine for conciseness:** Just give the direct answer.

*Self-Correction/Verification:*
- Apple reports segments as:
    - Products (iPhone, Mac, iPad, Wearables, Home & Accessories)
    - Services
- Under Products, iPhone is usually the largest single line item.
- Sometimes "Product

Tool calling

In [4]:
result = await run_tool_calling_agent_chat(
    client=client,
    model="default_model",
    user_prompt="Research AAPL focusing on financial health.",
    system_prompt=SYSTEM_PROMPT,
    tools=TOOL_SCHEMAS_CHAT,
    tool_functions=TOOL_FUNCTIONS,
    max_iterations=5,
    temperature=0.7,
    max_tokens=4096,
)

  [1] Reasoning: The user is asking me to research AAPL (Apple Inc.) with a focus on financial health. I need to gather relevant data usi...
  [1] Called get_financials(ticker='AAPL', statement_type='income', period='annual')
  [1] Called get_financials(ticker='AAPL', statement_type='balance_sheet', period='annual')
  [1] Called get_financials(ticker='AAPL', statement_type='cashflow', period='annual')
  [1] Called get_price_history(ticker='AAPL', period='1y', interval='1mo')
  [1] Called get_stock_news(ticker='AAPL')
  [1] Called get_recommendations(ticker='AAPL', months_back=12)
  [2] Reasoning: Good, I've got the data. Let me analyze what I have:  From the income statement (most recent: 2025-09-30): - Total Reven...
  [2] Called get_price_history(ticker='AAPL', period='6mo', interval='1mo')
  [3] Reasoning: Now I have more complete data. The most recent month (March 2026) shows: - Close: $248.80 (March 2026)  Let me compile a...
  [3] Agent finished — produced final response


In [5]:
# Results
n_tool_calls = sum(len(m.get("tool_calls", [])) for m in result["input"])
print(f"Tool calls: {n_tool_calls}")
print(f"Output length: {len(result['output_text'])} chars")

Tool calls: 7
Output length: 2626 chars


In [6]:
print(result['output_text'])

<think>
Now I have more complete data. The most recent month (March 2026) shows:
- Close: $248.80 (March 2026)

Let me compile all the financial health metrics:

From the data:
- Most recent EPS: $7.46 (FY 2025 ending Sep 30, 2025)
- Revenue: $416.161B (FY 2025)
- Net Income: $112.01B (FY 2025)
- Current Price: ~$248.80 (March 2026 close)
- Price change 1y: +17.59%
- Price change 6mo: -7.8% (from Oct 2025 to Mar 2026)

Key metrics for financial health:
- P/E = 248.80 / 7.46 = 33.35
- Debt/Equity = 98.657B / 73.733B = 1.34
- Working Capital: -$17.674B (negative, which is typical for Apple)
- Free Cash Flow: $98.767B
- FCF Margin = 98.767B / 416.161B = 23.7% (approx)

Market Cap estimation: 14.77326B shares × ~$248.80 = ~$3.68T

Now let me construct the equity research snapshot.
</think>

**AAPL (Apple Inc.)** | Consumer Discretionary | Market Cap ~$3.68T

**Key Metrics:**
- Revenue (TTM): $416.16B
- EPS (Diluted): $7.46
- P/E: ~33.4x
- Net Debt: $62.72B
- Debt/Equity: 1.34x
- Free Cash 

In [7]:
# Score it
reward = compute_reward(result["input"], VALID_TOOL_NAMES, result.get("reasoning_summaries"))
print(f"\nReward: {reward}")


Reward: {'valid_json': 1.0, 'thinking': 1.0, 'tool_selection': 2.0, 'efficiency': -2.0, 'completion': 1.0, 'no_hallucination': 0.0, 'total': 3.0}


In [8]:
result

{'input': [{'role': 'system',
   'content': "You are a sell-side equity research analyst producing a brief research snapshot.\n\nToday's date is 2026-03-29.\n\n## Instructions\n- Use the available tools to gather the data you need. Be efficient — call only what's necessary.\n- Think about what data points matter most for the given focus area before calling tools.\n- Thinking instructions: only LOW / CONCISE think is enabled or choose /nothink.\n- After gathering data, produce a concise equity research snapshot.\n\n## Output Format\nProduce a brief **Equity Research Snapshot** (~half page). Structure:\n\n**{COMPANY} ({TICKER})** | {Sector} | {Market Cap}\n\n**Key Metrics:** Revenue (TTM), EPS, P/E, margins, debt/equity — whatever is most relevant.\n**Recent Developments:** 1-3 bullet points on material news, earnings, or events.\n**Financial Highlights:** Key takeaways from the most recent financials.\n**Price Action:** Current price context, 52-week range, recent trend summary.\n**Anal

## Batch Async Baseline (MLX)

Fire 10 concurrent agent runs at the local mlx_lm.server to measure throughput and collect aggregate baseline metrics. Each agent makes multiple tool calls, so the real bottleneck is yfinance rate limiting — we use a semaphore to cap concurrent agents.

> **Server concurrency:** mlx_lm.server batches requests natively via `--decode-concurrency`
> (default 32) and `--prompt-concurrency` (default 8). No client-side batching needed for the
> LLM itself — the semaphore below only limits how many agents hit yfinance simultaneously.

In [9]:
import asyncio
import random
import time

from tqdm.asyncio import tqdm_asyncio

from bbb.helpers__data_gen import TICKERS, FOCUS_AREAS, make_user_prompt

# How many agents can run concurrently — limits yfinance pressure, not LLM throughput
CONCURRENCY = 5
N_TICKERS = 5

semaphore = asyncio.Semaphore(CONCURRENCY)
eval_tickers = random.sample(TICKERS, N_TICKERS)
print(f"Evaluating {N_TICKERS} tickers with concurrency={CONCURRENCY}: {eval_tickers}")

Evaluating 5 tickers with concurrency=5: ['KMB', 'ADBE', 'HUBS', 'LLY', 'GD']


In [10]:
async def run_one(ticker: str, focus: str) -> dict:
    """Run a single agent with semaphore-controlled concurrency."""
    async with semaphore:
        try:
            res = await run_tool_calling_agent_chat(
                client=client,
                model="default_model",
                user_prompt=make_user_prompt(ticker, focus),
                system_prompt=SYSTEM_PROMPT,
                tools=TOOL_SCHEMAS_CHAT,
                tool_functions=TOOL_FUNCTIONS,
                max_iterations=5,
                temperature=0.7,
                max_tokens=4096,
                verbose=False,
            )

            n_tool_calls = sum(len(m.get("tool_calls", [])) for m in res["input"])
            reward = compute_reward(res["input"], VALID_TOOL_NAMES, res.get("reasoning_summaries"))

            return {
                "ticker": ticker,
                "focus": focus,
                "n_tool_calls": n_tool_calls,
                "output_len": len(res["output_text"]),
                "reward": reward,
                "output_text": res["output_text"],
                "messages": res["input"],
            }
        except Exception as e:
            return {
                "ticker": ticker,
                "focus": focus,
                "error": str(e),
                "n_tool_calls": 0,
                "output_len": 0,
                "reward": {"total": -3.0},
                "output_text": "",
                "messages": [],
            }

In [11]:
tasks = [run_one(t, random.choice(FOCUS_AREAS)) for t in eval_tickers]

In [ ]:
t0 = time.time()
batch_results = await tqdm_asyncio.gather(*tasks, desc="Batch baseline")
elapsed = time.time() - t0

Batch baseline:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
errors = [r for r in batch_results if "error" in r]
print(f"\nDone: {len(batch_results)} tickers in {elapsed:.1f}s ({elapsed/len(batch_results):.1f}s/ticker)")
print(f"Errors: {len(errors)}")
if errors:
    for e in errors:
        print(f"  {e['ticker']}: {e['error']}")


Done: 10 tickers in 205.0s (20.5s/ticker)
Errors: 10
  TM: Connection error.
  DUK: Connection error.
  HOOD: Connection error.
  CCI: Connection error.
  BK: Connection error.
  APD: Connection error.
  LIN: Connection error.
  PEP: Connection error.
  PLD: Connection error.
  ECL: Connection error.


### Batch Results

In [ ]:
# Aggregate metrics
successful = [r for r in batch_results if "error" not in r]
rewards = [r["reward"]["total"] for r in successful]
tool_counts = [r["n_tool_calls"] for r in successful]
output_lens = [r["output_len"] for r in successful]

print(f"=== BATCH BASELINE ({len(successful)}/{len(batch_results)} successful) ===")
print(f"Throughput:   {elapsed:.1f}s total, {elapsed/len(batch_results):.1f}s/ticker")
print(f"Reward:       avg={sum(rewards)/len(rewards):.2f}  min={min(rewards):.2f}  max={max(rewards):.2f}")
print(f"Tool calls:   avg={sum(tool_counts)/len(tool_counts):.1f}  min={min(tool_counts)}  max={max(tool_counts)}")
print(f"Output chars: avg={sum(output_lens)/len(output_lens):.0f}")

components = ["valid_json", "thinking", "tool_selection", "efficiency", "completion", "no_hallucination"]
print("\n--- Per-component averages ---")
for comp in components:
    vals = [r["reward"].get(comp, 0) for r in successful]
    print(f"  {comp:>20}: {sum(vals)/len(vals):+.2f}")

print(f"\n{'Ticker':<8} {'Focus':<45} {'Reward':>7} {'Tools':>6} {'Output':>7}")
print("-" * 80)
for r in sorted(successful, key=lambda x: x["reward"]["total"]):
    rw = r["reward"]
    print(f"{r['ticker']:<8} {r['focus'][:43]:<45} {rw['total']:>+7.2f} {r['n_tool_calls']:>6} {r['output_len']:>7}")

=== BATCH BASELINE (0/10 successful) ===
Throughput:   205.0s total, 20.5s/ticker


ZeroDivisionError: division by zero

In [ ]:
# Save batch results
output_dir = PROJECT_ROOT / "data" / "bbb"
output_dir.mkdir(parents=True, exist_ok=True)

results_path = output_dir / "baseline_results_mlx.jsonl"
with open(results_path, "w") as f:
    for r in batch_results:
        record = {k: v for k, v in r.items() if k != "messages"}
        f.write(json.dumps(record, default=str) + "\n")

# Save best/worst full trajectories for talk examples
examples_path = output_dir / "baseline_examples_mlx.json"
worst = sorted(successful, key=lambda x: x["reward"]["total"])[:5]
best = sorted(successful, key=lambda x: x["reward"]["total"], reverse=True)[:3]

with open(examples_path, "w") as f:
    json.dump({"worst": worst, "best": best}, f, indent=2, default=str)

print(f"Saved {len(batch_results)} results to {results_path}")
print(f"Saved failure/success examples to {examples_path}")

## Databricks Baseline (Unsloth)

Everything below runs on Databricks with GPU. Uses Unsloth to load the model directly
and `run_local_agent_loop()` for inference (parses `<tool_call>` tags from raw model output).

Skip this section if testing locally with MLX above.

In [ ]:
%pip install -q unsloth trl

In [ ]:
import json
import sys
from pathlib import Path

from unsloth import FastLanguageModel

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "nb"))

from bbb.tools import TOOL_FUNCTIONS, TOOL_SCHEMAS
from bbb.helpers__data_gen import (
    SYSTEM_PROMPT,
    TICKERS,
    FOCUS_AREAS,
    make_user_prompt,
    _responses_tool_to_chat,
)
from bbb.helpers__inference import (
    run_local_agent_loop,
    compute_reward,
)

# Derive valid tool names from the tool registry (not hardcoded)
VALID_TOOL_NAMES = set(TOOL_FUNCTIONS.keys())

In [ ]:
# Load Qwen3-4B — no fine-tuning, just the base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B",
    max_seq_length=8192,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print(f"Model loaded on {model.device}")

## Tool Setup

Convert tool schemas from Responses API (flat) to Chat Completions (nested) format,
which is what Qwen3's `apply_chat_template(tools=...)` expects.

In [ ]:
from bbb.tools import TOOL_SCHEMAS

# Convert flat Responses API schemas to nested Chat Completions format
TOOLS_CHAT = [_responses_tool_to_chat(t) for t in TOOL_SCHEMAS]

for tool in TOOLS_CHAT:
    fn = tool["function"]
    params = list(fn["parameters"]["properties"].keys())
    print(f"  {fn['name']}({', '.join(params)})")

## Single Ticker Test

Run one ticker with verbose output to see exactly what the raw model does.

In [ ]:
result = run_local_agent_loop(
    model=model,
    tokenizer=tokenizer,
    user_prompt="Research AAPL focusing on financial health and balance sheet strength.",
    system_prompt=SYSTEM_PROMPT,
    tools_chat=TOOLS_CHAT,
    tool_functions=TOOL_FUNCTIONS,
    max_iterations=8,
    enable_thinking=True,
    verbose=True,
)

print(f"\nIterations: {result['n_iterations']}")
print(f"Tool calls: {result['n_tool_calls']}")
print(f"Final output: {len(result['output_text'])} chars")

In [ ]:
# Show the full message flow
for i, msg in enumerate(result["messages"]):
    role = msg["role"].upper()
    content = msg.get("content", "") or ""
    tc = msg.get("tool_calls", [])

    if role == "SYSTEM":
        print(f"[{i}] {role}: {content[:80]}...")
    elif role == "USER":
        print(f"[{i}] {role}: {content}")
    elif role == "ASSISTANT" and tc:
        names = [t["function"]["name"] for t in tc]
        has_think = "<think>" in content
        print(f"[{i}] {role} (think={has_think}, tools={names})")
        if content:
            print(f"    content: {content[:150]}...")
    elif role == "ASSISTANT":
        print(f"[{i}] {role} (final): {content[:200]}...")
    elif role == "TOOL":
        print(f"[{i}] {role}: {len(content)} chars")
    print()

In [ ]:
# Score this trajectory
reward = compute_reward(result["messages"], VALID_TOOL_NAMES)
print("Reward components:")
for k, v in reward.items():
    print(f"  {k:>20}: {v}")

## Batch Evaluation

Run on ~20 tickers to get aggregate baseline metrics.

In [ ]:
import random

# Pick 20 diverse tickers
eval_tickers = random.sample(TICKERS, 20)
print(f"Evaluating: {eval_tickers}")

In [ ]:
baseline_results = []

for ticker in eval_tickers:
    focus = random.choice(FOCUS_AREAS)
    prompt = make_user_prompt(ticker, focus)

    print(f"\n{'='*50}")
    print(f"{ticker} — {focus}")

    try:
        res = run_local_agent_loop(
            model=model,
            tokenizer=tokenizer,
            user_prompt=prompt,
            system_prompt=SYSTEM_PROMPT,
            tools_chat=TOOLS_CHAT,
            tool_functions=TOOL_FUNCTIONS,
            max_iterations=8,
            enable_thinking=True,
            verbose=True,
        )

        reward = compute_reward(res["messages"], VALID_TOOL_NAMES)

        baseline_results.append({
            "ticker": ticker,
            "focus": focus,
            "n_tool_calls": res["n_tool_calls"],
            "n_iterations": res["n_iterations"],
            "output_len": len(res["output_text"]),
            "reward": reward,
            "messages": res["messages"],
        })

        print(f"  → reward={reward['total']}, tool_calls={res['n_tool_calls']}, output={len(res['output_text'])} chars")

    except Exception as e:
        print(f"  → ERROR: {e}")
        baseline_results.append({
            "ticker": ticker,
            "focus": focus,
            "error": str(e),
            "reward": {"total": -3.0},
        })

print(f"\nCompleted: {len(baseline_results)}/{len(eval_tickers)}")

## Results Summary

In [ ]:
# Aggregate metrics
rewards = [r["reward"]["total"] for r in baseline_results]
tool_counts = [r.get("n_tool_calls", 0) for r in baseline_results]
output_lens = [r.get("output_len", 0) for r in baseline_results]
completions = [r["reward"].get("completion", 0) for r in baseline_results]
valid_jsons = [r["reward"].get("valid_json", 0) for r in baseline_results]

print("=== BASELINE RESULTS ===")
print(f"Reward:       avg={sum(rewards)/len(rewards):.2f}  min={min(rewards)}  max={max(rewards)}")
print(f"Tool calls:   avg={sum(tool_counts)/len(tool_counts):.1f}  min={min(tool_counts)}  max={max(tool_counts)}")
print(f"Output chars: avg={sum(output_lens)/len(output_lens):.0f}")
print(f"Completion rate: {sum(completions)/len(completions)*100:.0f}%")
print(f"Valid JSON rate:  {sum(valid_jsons)/len(valid_jsons)*100:.0f}%")

print("\n--- Per-component averages ---")
components = ["valid_json", "thinking", "tool_selection", "efficiency", "completion", "no_hallucination"]
for comp in components:
    vals = [r["reward"].get(comp, 0) for r in baseline_results]
    print(f"  {comp:>20}: {sum(vals)/len(vals):+.2f}")

In [ ]:
# Per-ticker breakdown
print(f"{'Ticker':<8} {'Reward':>7} {'Tools':>6} {'Output':>7} {'JSON':>5} {'Done':>5}")
print("-" * 45)
for r in sorted(baseline_results, key=lambda x: x["reward"]["total"]):
    rw = r["reward"]
    print(
        f"{r['ticker']:<8} {rw['total']:>+7.2f} "
        f"{r.get('n_tool_calls', 0):>6} "
        f"{r.get('output_len', 0):>7} "
        f"{'Y' if rw.get('valid_json') else 'N':>5} "
        f"{'Y' if rw.get('completion') else 'N':>5}"
    )

## Save Baseline Results

Save for comparison with SFT and RL models.

In [ ]:
# Save results (without full messages to keep file small)
output_path = PROJECT_ROOT / "data" / "bbb" / "baseline_results.jsonl"

with open(output_path, "w") as f:
    for r in baseline_results:
        record = {k: v for k, v in r.items() if k != "messages"}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(baseline_results)} results to {output_path}")

# Also save a few full trajectories (failure examples for the talk)
examples_path = PROJECT_ROOT / "data" / "bbb" / "baseline_examples.json"
worst = sorted(baseline_results, key=lambda x: x["reward"]["total"])[:5]
best = sorted(baseline_results, key=lambda x: x["reward"]["total"], reverse=True)[:3]

with open(examples_path, "w") as f:
    json.dump({"worst": worst, "best": best}, f, indent=2, default=str)

print(f"Saved failure/success examples to {examples_path}")